# 07 — Cifras del informe

Recalcula, desde `resultados/` y los parquet por comuna, las cifras de `docs/informe/informe_modelo_27.tex` que no salen directamente de otro notebook. Cada celda corresponde a una sección del informe e imprime sus cifras con la sección donde se usan.

| Sección del informe | Celda | Otras fuentes |
|---|---|---|
| 2. Fuentes de datos y cobertura | S2 | `05_pais.ipynb` (E1) genera `diagnostico.csv` |
| 3. El modelo (perfiles, Puente Alto) | S3 | `03_perfiles_27.ipynb` (D4c), `05_pais.ipynb` (E3) |
| 3.4 La priori | S3b | `01_poc_historica.ipynb`, `02_espina.ipynb` (C0, C4), `03_perfiles_27.ipynb` (D3c) |
| 4. Las variables (personas, hogares y viviendas) | — | `manzanas/inventario.py` y `manzanas/unidades.py`; verificación en `03` (D2), `05` (E1) y `08_hogares_viviendas.ipynb` (H0b, H1, H4) |
| 5 y 6. Categorías que no calzan y el resto | S5-6 | |
| 7. Conteos enmascarados | S7 | `02_espina.ipynb` (C3) para la espina |
| 8. Personas que "sobran" | S8 | |
| 9. Verificaciones y corrida nacional | S9 | `05_pais.ipynb` (E5) |
| 10. Validación | S10 | `06_validacion_provincias.ipynb` |

Requiere haber corrido `05_pais.ipynb` (diagnóstico, piloto y corrida nacional) y `06_validacion_provincias.ipynb`. No ajusta modelos; S5-6 recorre las 295 comunas y tarda unos minutos.

In [1]:
import os

import numpy as np
import pandas as pd

from manzanas.datos import cargar_comuna, ruta_comuna
from manzanas.informe import estadisticas_comunas
from manzanas.inventario import construir_inventario
from manzanas.ipf import ipf_perfiles
from manzanas.pais import RUTA_COMUNAS, RUTA_MODELOS, RUTA_PAIS
from manzanas.perfiles import cargar_modelo, codigos_bloque, proyectar
from manzanas.provincias import RUTA_VALIDACION

pd.set_option("display.width", 250)
diag = pd.read_csv(os.path.join(RUTA_PAIS, "diagnostico.csv"), index_col=0)
ok = diag["inventario"] == "ok"
cuts_ok = diag.index[ok].tolist()
resumen = pd.read_csv(os.path.join(RUTA_PAIS, "resumen_pais.csv")).drop_duplicates("cut", keep="last").set_index("cut")
print(f"comunas en el diagnóstico: {len(diag)} | con inventario completo: {ok.sum()} | en la corrida nacional: {len(resumen)}")

comunas en el diagnóstico: 346 | con inventario completo: 295 | en la corrida nacional: 295


### S2. Fuentes de datos y cobertura

In [2]:
# S2. Sección 2 del informe
filas_mz = sum(len(pd.read_parquet(ruta_comuna("manzanas", c, RUTA_COMUNAS), columns=["MANZENT"])) for c in diag.index)
n_cols_mz = len(pd.read_parquet(ruta_comuna("manzanas", 13108, RUTA_COMUNAS)).columns)
bajo = diag["bajo_umbral"] == 1
print(f"registros de personas (346 comunas): {diag['personas'].sum():,}")
print(f"base manzana-entidad: {filas_mz:,} filas, {n_cols_mz} columnas")
print(f"comunas: {len(diag)} | inventario completo: {ok.sum()} | bajo umbral: {bajo.sum()} "
      f"({diag.loc[bajo, 'personas'].sum():,} personas, {100 * diag.loc[bajo, 'personas'].sum() / diag['personas'].sum():.1f}%)")
print(f"cobertura de la base: {diag.loc[ok, 'personas'].sum():,} personas "
      f"({100 * diag.loc[ok, 'personas'].sum() / diag['personas'].sum():.1f}%), {diag.loc[ok, 'manzanas'].sum():,} unidades con población")
print("personas del microdato = suma de n_per en todas las comunas:", bool((diag["personas"] == diag["suma_n_per"]).all()))
print("comunas que fallan el inventario y no son bajo umbral:", int((~ok & ~bajo).sum()))

registros de personas (346 comunas): 18,480,432
base manzana-entidad: 197,032 filas, 212 columnas
comunas: 346 | inventario completo: 295 | bajo umbral: 51 (151,199 personas, 0.8%)
cobertura de la base: 18,329,233 personas (99.2%), 188,175 unidades con población
personas del microdato = suma de n_per en todas las comunas: True
comunas que fallan el inventario y no son bajo umbral: 0


### S3. El modelo: perfiles y convergencia

In [3]:
# S3. Sección 3 del informe
m13108 = cargar_modelo(os.path.join(RUTA_MODELOS, "modelo_13108.npz"))
print(f"Independencia: {len(m13108['mu_p']):,} perfiles | de una sola persona: {100 * (m13108['mu_p'] == 1).mean():.0f}%")
pa = resumen.loc[13201]
print(f"Puente Alto: {pa['celdas_W'] / 1e6:.0f} M celdas | {int(pa['iteraciones'])} iteraciones | {pa['segundos_ajuste'] / 60:.0f} min "
      "(tolerancia relativa)")
print("el intento con tolerancia absoluta (interrumpido tras 4,6 h) está documentado en 05_pais.ipynb, E3")

Independencia: 40,946 perfiles | de una sola persona: 75%
Puente Alto: 541 M celdas | 4650 iteraciones | 17 min (tolerancia relativa)
el intento con tolerancia absoluta (interrumpido tras 4,6 h) está documentado en 05_pais.ipynb, E3


### S3b. La priori: evidencia del volumen de edificios (sección 3.4)


In [2]:
# S3b. Sección 3.4 del informe: por qué la priori de volumen de edificios no ayudó (Independencia)
import geopandas as gpd
w = gpd.read_parquet(os.path.join("priori", "manzanas_weights.parquet"))
w = pd.DataFrame({"MANZENT": w["MANZENT"].astype(np.int64), "weight": w["weight"]})
mz = cargar_comuna(13108, ruta_resultados=RUTA_COMUNAS).manzanas.merge(w, on="MANZENT", how="left")
print(f"manzanas con población: {len(mz)} | sin volumen: {mz['weight'].isna().sum()} | con volumen 0: {(mz['weight'] == 0).sum()}")
mz = mz[mz["weight"].notna()].copy()
tramos = ["0_5", "6_13", "14_17", "18_24", "25_44", "45_59", "60_mas"]
pct = pd.DataFrame({t: pd.to_numeric(mz[f"n_edad_{t}"].astype(str), errors="coerce") / mz["n_per"] for t in tramos})
densidad = mz["weight"] / mz["n_per"]
prom_edad = mz["prom_edad"].astype(str).str.replace(",", ".", regex=False).astype(float)
print(f"corr(volumen, n_per) = {np.corrcoef(mz['weight'], mz['n_per'])[0, 1]:.2f}")
print("corr con el % de cada tramo de edad:")
print(pd.DataFrame({"volumen por persona": [np.corrcoef(densidad, pct[t])[0, 1] for t in tramos],
                    "prom_edad": [np.corrcoef(prom_edad, pct[t])[0, 1] for t in tramos]}, index=tramos).round(2).T.to_string())
print("SRMSE de la edad por manzana sin restringirla: costo con prom_edad 0,538 vs sin costo 0,794; con 3 variables 0,345 vs 0,381 "
      "agregando el volumen -> 01_poc_historica.ipynb (Parte 0 y priori); 0,794/0,538/0,345 recalculados en 02_espina.ipynb (C0)")
print("semilla con prom_edad vs uniforme con la edad restringida: 5,6e-12 (02_espina, C4) | semilla heredada vs uniforme: 2,4e-11 (03_perfiles_27, D3c)")

manzanas con población: 469 | sin volumen: 1 | con volumen 0: 2
corr(volumen, n_per) = 0.94
corr con el % de cada tramo de edad:
                      0_5  6_13  14_17  18_24  25_44  45_59  60_mas
volumen por persona -0.10 -0.18  -0.16  -0.15  -0.09   0.24    0.11
prom_edad           -0.59 -0.55  -0.26  -0.39  -0.71   0.14    0.95
SRMSE de la edad por manzana sin restringirla: costo con prom_edad 0,538 vs sin costo 0,794; con 3 variables 0,345 vs 0,381 agregando el volumen -> 01_poc_historica.ipynb (Parte 0 y priori); 0,794/0,538/0,345 recalculados en 02_espina.ipynb (C0)
semilla con prom_edad vs uniforme con la edad restringida: 5,6e-12 (02_espina, C4) | semilla heredada vs uniforme: 2,4e-11 (03_perfiles_27, D3c)


### S5-6. Categorías que no calzan y el resto

Recorre las 295 comunas: para cada bloque cuenta, por código del microdato, a las personas que no caen en ninguna categoría publicada (el resto), y para cada columna `n_*` el enmascaramiento. Guarda los resultados en `resultados/pais/informe_*.csv`.

In [4]:
# S5-6. Secciones 5 y 6 del informe (tarda unos minutos)
columnas, resto, contenedor = estadisticas_comunas(cuts_ok)
columnas.to_csv(os.path.join(RUTA_PAIS, "informe_columnas.csv"))
resto.to_csv(os.path.join(RUTA_PAIS, "informe_resto.csv"), index=False)
contenedor.drop(columns="ids").to_csv(os.path.join(RUTA_PAIS, "informe_contenedor.csv"))

N = diag.loc[ok, "personas"].sum()
tabla_resto = resto.pivot_table(index="bloque", columns="codigo", values="personas", aggfunc="sum", fill_value=0)
print("Tabla 'Composición del resto' (personas por código del microdato):")
print(tabla_resto.loc[[b for b in columnas.index.get_level_values(0).unique() if b in tabla_resto.index]].to_string())

def r(bloque, codigo):
    fila = resto[(resto["bloque"] == bloque) & (resto["codigo"] == codigo)]
    return int(fila["personas"].sum())

analf_5_14 = sum(int(((p["p37_alfabet"] == 2) & (p["edad"] < 15)).sum())
                 for p in (pd.read_parquet(ruta_comuna("personas", c, RUTA_COMUNAS), columns=["p37_alfabet", "edad"]) for c in cuts_ok))
print(f"\nSección 5: doble nacionalidad (código 2) = {r('nacionalidad', '2'):,} | CIUO 999 = {r('ciuo', '999'):,} | "
      f"CAENES 999 = {r('caenes', '999'):,} | analfabetos de 5 a 14 años = {analf_5_14:,}")
no_resp = resto[resto["codigo"] == "-99"].groupby("bloque")["personas"].sum()
print(f"Sección 6: no respuesta (-99) entre {100 * no_resp.min() / N:.2f}% y {100 * no_resp.max() / N:.2f}% de las personas, según el bloque")

Tabla 'Composición del resto' (personas por código del microdato):
codigo             -99         1         2     999        NA
bloque                                                      
inmigrante      112443  16614305         0       0         0
nacionalidad    100883  16500963    242909       0         0
pueblo          106874         0  16152411       0         0
afro            109754         0  18046920       0         0
lengua          102478         0  16862405       0    863839
religion         87058         0   3880472       0   3249100
dif_ver          98314  13800725   2728649       0    863839
dif_oir         100302  16016425    954105       0    863839
dif_mover       102771  15169492   1293982       0    863839
dif_cogni       104339  15078207   1789018       0    863839
dif_cuidado     101695  16742842    321615       0    863839
dif_comunic      99670  16863671    304236       0    863839
discapacidad     97433         0  15437282       0    863839
estado_civil     5

### S7. Conteos enmascarados

In [5]:
# S7. Sección 7 del informe
unidades = diag.loc[ok, "manzanas"].sum()
sensibles = ["inmigrante", "nacionalidad", "pueblo", "afro", "lengua", "religion"]
c = columnas.reset_index()
s = c[c["bloque"].isin(sensibles)].assign(pct_manzanas=lambda d: 100 * d["manzanas_con_asterisco"] / unidades,
                                         pct_residuo=lambda d: 100 * d["residuo"] / d["total"])
print("Tabla 'Enmascaramiento':")
print(s[["columna", "total", "manzanas_con_asterisco", "pct_manzanas", "residuo", "pct_residuo"]].round(1).to_string(index=False))
otras = c[~c["bloque"].isin(sensibles)]
print(f"{len(otras)} columnas restantes: entre {otras['manzanas_con_asterisco'].min()} y {otras['manzanas_con_asterisco'].max()} "
      f"manzanas con '*' por columna (mediana {otras['manzanas_con_asterisco'].median():.0f}) | residuo total {otras['residuo'].sum():,.0f} personas "
      f"| columnas con algún '*' en el país: {(c['manzanas_con_asterisco'] > 0).sum()} de {len(c)}")

d = resumen["dif_enmascaradas_pct"]
peor = resumen.loc[d.idxmax()]
print(f"\ndiferencia estimado vs residuo en manzanas enmascaradas (% del total comunal de la categoría): mediana {d.median():.3f}% "
      f"| percentil 95 {d.quantile(0.95):.2f}% | máximo {d.max():.2f}% ({peor['comuna']}, {peor['dif_enmascaradas']:.1f} personas)")

# Independencia: inmigrantes estimados por manzana enmascarada, en el modelo de 27 bloques
inv_i = construir_inventario(cargar_comuna(13108, ruta_resultados=RUTA_COMUNAS))
_, nu_inm = codigos_bloque(inv_i, "inmigrante")
masc = np.isnan(nu_inm[0])
est = proyectar(m13108, "inmigrante", nu_inm.shape[0])[0, masc]
print(f"Independencia (27 bloques): {masc.sum()} manzanas con n_inmigrantes enmascarado | estimado por manzana entre "
      f"{est.min():.1f} y {est.max():.1f} | suma {est.sum():.2f}")

# Ejemplo del informe: 4 manzanas de 100 personas, inmigrantes enmascarados en C y D
perf = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])            # (edad: 0 adulto, 1 joven; inmigrante: 0 sí, 1 resto)
mu = np.array([60, 140, 10, 190])
adultos, n_per = np.array([70, 30, 80, 20]), np.full(4, 100)
inm = np.array([40, 20, np.nan, np.nan])
W, it, conv = ipf_perfiles(mu, [perf[:, 0], perf[:, 1]], [np.vstack([adultos, n_per - adultos]), np.vstack([inm, n_per - inm])])
print("\nejemplo A-D, inmigrantes estimados:", dict(zip("ABCD", W[perf[:, 1] == 0].sum(axis=0).round(2))), "| convergió:", conv)

Tabla 'Enmascaramiento':
           columna    total  manzanas_con_asterisco  pct_manzanas  residuo  pct_residuo
     n_inmigrantes  1602485                   49197          26.1  85198.0          5.3
    n_nacionalidad  1484478                   43123          22.9  75235.0          5.1
    n_pueblos_orig  2069948                   50194          26.7 103139.0          5.0
n_afrodescendencia   172559                   35549          18.9  52142.0         30.2
 n_lengua_indigena   500511                   66183          35.2 109437.0         21.9
        n_religion 11112603                    1010           0.5   6580.0          0.1
76 columnas restantes: entre 1 y 214 manzanas con '*' por columna (mediana 5) | residuo total 2,000 personas | columnas con algún '*' en el país: 82 de 82

diferencia estimado vs residuo en manzanas enmascaradas (% del total comunal de la categoría): mediana 0.007% | percentil 95 0.12% | máximo 0.45% (PUERTO OCTAY, 3.2 personas)
Independencia (27 bloques): 

### S8. Personas que "sobran"

In [6]:
# S8. Sección 8 del informe
print(f"contenedor comunal: {contenedor['unidades'].sum()} unidades con población en {(contenedor['unidades'] > 0).sum()} comunas, "
      f"{contenedor['personas'].sum():,} personas ({100 * contenedor['personas'].sum() / N:.1f}%)")
mz_i = cargar_comuna(13108, ruta_resultados=RUTA_COMUNAS).manzanas
print("Independencia:", mz_i.loc[mz_i["CONTENEDOR_COMUNAL"] == 1, ["MANZENT", "n_per"]].to_dict("records"))
print(f"comunas bajo umbral: {(diag['bajo_umbral'] == 1).sum()} con {diag.loc[diag['bajo_umbral'] == 1, 'personas'].sum():,} personas")

contenedor comunal: 331 unidades con población en 295 comunas, 251,215 personas (1.4%)
Independencia: [{'MANZENT': 13108991999999, 'n_per': 1781}]
comunas bajo umbral: 51 con 151,199 personas


### S9. Verificaciones y corrida nacional

In [7]:
# S9. Sección 9 del informe
listas = resumen[resumen["error"].isna()]
print(f"comunas: {len(listas)} sin fallas de {len(resumen)} | convergieron: {listas['convergio'].astype(bool).sum()} "
      f"| tiempo total {listas['segundos_total'].sum() / 3600:.1f} h")
it_max = listas.loc[listas["iteraciones"].idxmax()]
print(f"iteraciones: mediana {listas['iteraciones'].median():.0f} | máximo {int(it_max['iteraciones'])} ({it_max['comuna']})")
print(f"error marginal máximo: {listas['error_max'].max():.3f} personas (relativo a lo permitido {listas['error_rel_max'].max():.2f})")
print(f"base de cruces: {listas['filas_cruces'].sum() / 1e6:.0f} M filas, {listas['MB_cruces'].sum() / 1e3:.1f} GB | modelos: {listas['MB_modelo'].sum() / 1e3:.1f} GB")
p = listas["pct_1_persona_>50%_en_1_manzana"]
unicos = listas["perfiles_1_persona"].sum()
concentrados = (listas["perfiles_1_persona"] * p / 100).sum()
print(f"privacidad: {unicos / 1e6:.1f} M perfiles de una persona | con >50% en una manzana: {100 * concentrados / unicos:.2f}% "
      f"| comunas sobre 5%: {(p > 5).sum()}")
print(listas.assign(pct=p).sort_values("pct", ascending=False).head(4)[["comuna", "manzanas", "pct"]].round(1).to_string())

comunas: 295 sin fallas de 295 | convergieron: 295 | tiempo total 4.1 h
iteraciones: mediana 1700 | máximo 6550 (LA FLORIDA)
error marginal máximo: 0.996 personas (relativo a lo permitido 1.00)
base de cruces: 520 M filas, 2.9 GB | modelos: 31.7 GB
privacidad: 4.1 M perfiles de una persona | con >50% en una manzana: 0.71% | comunas sobre 5%: 21
           comuna  manzanas  pct
cut                             
8314  ALTO BIOBÍO        54  9.4
3303     FREIRINA       194  9.2
1405         PICA       113  8.6
4202       CANELA       268  8.3


### S10. Validación con verdad conocida

In [8]:
# S10. Sección 10 del informe (desde los archivos que guarda 06_validacion_provincias.ipynb)
cruces = pd.read_parquet(os.path.join(RUTA_VALIDACION, "errores_cruces.parquet"))
prov = pd.read_csv(os.path.join(RUTA_VALIDACION, "resumen_provincias.csv"), index_col=0)
print(f"provincias: {len(prov)} | comunas: {cruces['comuna'].nunique()} | personas: {prov['personas'].sum():,}")
for m in ["modelo", "raking_2", "provincial", "independencia"]:
    print(f"  personas mal ubicadas, {m}: {100 * cruces[f'err_{m}'].sum() / (2 * cruces['n'].sum()):.2f}%")
por_par = cruces.groupby(["var1", "var2"])[["err_modelo", "err_raking_2", "n"]].sum()
validos = 100 * por_par["err_raking_2"] / (2 * por_par["n"]) >= 0.01
mejora = 100 * (1 - por_par["err_modelo"] / por_par["err_raking_2"])[validos]
print(f"el modelo mejora al raking de 2 bloques en {(mejora > 0).sum()} de {validos.sum()} cruces (reducción mediana {mejora.median():.0f}%)")
pc = cruces.groupby("comuna")[["err_modelo", "n"]].sum()
pc["error"], pc["personas"] = 100 * pc["err_modelo"] / (2 * pc["n"]), pc["n"] / len(por_par)
print("error del modelo por tamaño de comuna (mediana):",
      pc.groupby(pd.cut(pc["personas"], [0, 10_000, 150_000, 1e7], labels=["<10 mil", "10-150 mil", ">150 mil"]), observed=True)["error"]
      .median().round(2).to_dict())

provincias: 43 | comunas: 287 | personas: 17,808,383
  personas mal ubicadas, modelo: 0.18%
  personas mal ubicadas, raking_2: 0.25%
  personas mal ubicadas, provincial: 3.52%
  personas mal ubicadas, independencia: 4.48%
el modelo mejora al raking de 2 bloques en 250 de 287 cruces (reducción mediana 21%)
error del modelo por tamaño de comuna (mediana): {'<10 mil': 0.34, '10-150 mil': 0.24, '>150 mil': 0.14}
